# Vision Transformer desde cero en CUDA — MNIST

Notebook de ejecución para Google Colab con GPU.

**Antes de empezar:** menú `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU (T4)**.

El orden de las celdas es deliberado y hay que respetarlo:

1. Comprobar la GPU
2. Subir el proyecto
3. Verificar los kernels **en CPU** (no necesita GPU, detecta errores de índice antes de compilar)
4. Compilar
5. Comprobar la carga de datos (ASCII art)
6. **Corrida de prueba de una sola configuración** — confirmar que el tiempo por época es razonable
7. Sólo entonces: la batería completa de experimentos
8. Generar figuras y tablas, y descargar los resultados

El paso 6 no es opcional: lanzar las corridas completas a ciegas es la forma más rápida de perder media hora.

## 1. Comprobar la GPU asignada

In [ ]:
!nvidia-smi
!nvcc --version

# La arquitectura correcta para compilar depende de la GPU que toque:
#   T4  -> sm_75   (la habitual en Colab gratuito)
#   P100-> sm_60
#   V100-> sm_70
#   A100-> sm_80
#   L4  -> sm_89
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
ARCH = {'T4':'sm_75','P100':'sm_60','V100':'sm_70','A100':'sm_80','L4':'sm_89'}
arch = next((v for k, v in ARCH.items() if k in name), 'sm_75')
print(f'GPU detectada: {name}  ->  compilar con ARCH={arch}')
%env ARCH=$arch

## 2. Subir el proyecto

Dos opciones. **Opción A (Drive)**: sube la carpeta `VisionTransformer/` completa a tu Google Drive y ejecuta la primera celda. **Opción B (zip)**: comprime la carpeta y súbela directamente.

En cualquier caso, **no subas `data/mnist_processed.npz`** (220 MB y no se usa: el pipeline lee los `.ubyte`).

In [ ]:
# --- Opción A: desde Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/VisionTransformer'   # ajusta la ruta si hace falta
%cd $PROJECT
!ls -la && ls -la data

In [ ]:
# --- Opción B: subir un zip (ejecuta ESTA celda en lugar de la anterior) ---
# from google.colab import files
# up = files.upload()                     # selecciona VisionTransformer.zip
# !unzip -q -o VisionTransformer.zip -d /content/
# PROJECT = '/content/VisionTransformer'
# %cd $PROJECT
# !ls -la && ls -la data

## 3. Verificar los kernels en CPU

Emula las mismas expresiones de índice de los kernels tileados y comprueba los gradientes analíticos contra diferencias numéricas. No usa la GPU: si algo falla aquí, no tiene sentido compilar.

In [ ]:
!make verify

## 4. Compilar

In [ ]:
!make clean && make ARCH=$ARCH 2>&1 | tail -30
!ls -la bin/

## 5. Comprobar la carga de datos

Si los dígitos no se ven, el parser de la cabecera IDX está mal y no hay que seguir. La media de píxeles debe salir 0.1307.

In [ ]:
!./bin/test_loader data 2>&1 | head -50

## 6. Corrida de prueba (una sola configuración)

Dos épocas con parche 7x7. **Mira el tiempo de la primera época**: debe estar bastante por debajo de 30 s. Si lo supera, el programa reduce el subconjunto a la mitad y reinicia solo, avisando de lo que cambió y por qué.

In [ ]:
!./bin/vit_train --data-dir data --out-dir results --tag smoke_p7_shared \
                 --patch 7 --attn-mem shared --epochs 2 \
                 --train-n 5000 --test-n 2000 --batch 64 --lr 1e-3

### Comparación con la referencia en PyTorch

Control de sanidad opcional: entrena la misma arquitectura con PyTorch. Si la versión CUDA queda muy por debajo de estos números, el problema está en los kernels.

In [ ]:
!python3 scripts/vit_reference.py --patch 7 --epochs 15

## 7. Batería completa de experimentos

Sólo si el paso 6 salió bien. Son 3 entrenamientos completos (uno por tamaño de parche), el microbenchmark aislado de memoria global vs compartida, y 3 corridas cortas con memoria global para el tiempo extremo a extremo. Presupuesto total: unos 15-25 minutos en una T4.

In [ ]:
!bash scripts/run_experiments.sh 2>&1 | tail -80

## 8. Figuras, tablas y descarga de resultados

In [ ]:
!pip install -q matplotlib
!python3 scripts/make_report_assets.py
!ls -la informe/figs informe/tables results

In [ ]:
from IPython.display import Image, display
for f in ('fig_patch_tradeoff.png', 'fig_curves.png', 'fig_memory.png'):
    display(Image(f'informe/figs/{f}'))

In [ ]:
# Empaqueta todo lo que hace falta para cerrar el informe en la máquina local.
!zip -qr resultados_vit.zip results informe/figs informe/tables
from google.colab import files
files.download('resultados_vit.zip')

---

## Qué hacer con el zip

Descomprímelo sobre la carpeta local del proyecto (sobrescribiendo `results/`, `informe/figs/` e `informe/tables/`) y compila el informe:

```bash
unzip -o resultados_vit.zip
python3 scripts/make_report_assets.py
tectonic informe/informe.tex
```

Las cifras citadas en el texto del informe se actualizan solas: `make_report_assets.py` genera `informe/tables/kpi.tex` con los valores medidos y el `.tex` los lee de ahí.